In [1]:
# To Supress Unnecessary Warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Data Processing
import numpy as np
import pandas as pd

# Utilities
from sklearn.preprocessing import StandardScaler
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
DATA_PATH = "../Data/OldSensorNodeData.csv"

OUTLIER_COLUMNS = ['AQI', 'Temp', 'Hum']

INPUT_FEATURES = ['AQI', 'Temp', 'Hum', 'Pres', 'hour_sin', 'hour_cos']
TARGET_FEATURES = ['AQI', 'Temp', 'Hum']

SEQ_LEN = 120
HORIZON_ORDER = ['1min', '5min', '15min']
HORIZONS = {
    '1min': 12,
    '5min': 60,
    '15min': 180
}
TRAINING_DATA_PERCENTAGE = 0.8

SCALED_INPUT_FEATURES = ['AQI', 'Temp', 'Hum']
UNSCALED_INPUT_FEATURES = ['Pres', 'hour_sin', 'hour_cos']

OFFLINE_LEARNING_RATE = 0.001

In [3]:
def cap_outliers_iqr(df, columns):
    bounds = {}
    df = df.copy()

    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(lower=lower, upper=upper)

        bounds[col] = {
            "lower": lower,
            "upper": upper
        }
    return df, bounds

In [4]:
def process_raw_sensor_data(df):
    # Capping Outlier Columns
    df, bounds = cap_outliers_iqr(df, OUTLIER_COLUMNS)

    # Add Hourly Features
    df['hour_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.hour / 24)

    return df, bounds

In [5]:
def build_lstm_sequences(df):
    X, y = [], []

    x_data = df[INPUT_FEATURES].values
    y_data = df[TARGET_FEATURES].values

    max_horizon = max(HORIZONS.values())

    for i in range(SEQ_LEN, len(df) - max_horizon):
        # ---- Input sequence ----
        X.append(x_data[i - SEQ_LEN:i])

        # ---- Target vector ----
        target = []
        for col_idx in range(len(TARGET_FEATURES)):
            for h in HORIZON_ORDER:
                target.append(y_data[i + HORIZONS[h], col_idx])

        y.append(target)

    return np.array(X), np.array(y)

In [6]:
def split_train_val_data(X, y):
    # Split data into train and test sets
    split_idx = int(len(X) * TRAINING_DATA_PERCENTAGE)

    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]
    return X_train, X_val, y_train, y_val

In [7]:
def scale_inputs(X_train, X_val):
    scaled_feature_indices = [INPUT_FEATURES.index(f) for f in SCALED_INPUT_FEATURES]

    input_scalers = []
    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()

    for idx in scaled_feature_indices:
        scaler = StandardScaler()

        # Fit on training input feature idx (across all timesteps)
        X_train_scaled[:, :, idx] = scaler.fit_transform(
            X_train[:, :, idx].reshape(-1, 1)
        ).reshape(X_train.shape[0], X_train.shape[1])

        # Transform validation input feature idx
        X_val_scaled[:, :, idx] = scaler.transform(
            X_val[:, :, idx].reshape(-1, 1)
        ).reshape(X_val.shape[0], X_val.shape[1])

        input_scalers.append(scaler)
    return X_train_scaled, X_val_scaled, input_scalers

In [8]:
def scale_targets(y_train, y_val):
    target_scalers = []
    y_train_scaled = y_train.copy()
    y_val_scaled = y_val.copy()

    num_targets = y_train.shape[1]

    for i in range(num_targets):
        scaler = StandardScaler()

        # Fit on training target i
        y_train_scaled[:, i] = scaler.fit_transform(
            y_train[:, i].reshape(-1, 1)
        ).flatten()

        # Transform validation target i
        y_val_scaled[:, i] = scaler.transform(
            y_val[:, i].reshape(-1, 1)
        ).flatten()

        target_scalers.append(scaler)

    return y_train_scaled, y_val_scaled, target_scalers

In [9]:
# Load dataset
data = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])

# Process raw sensor data
data, cap_iqr_bounds = process_raw_sensor_data(data)

# Build LSTM sequences
X, y = build_lstm_sequences(data)

# Split data into train and validation sets
X_train, X_val, y_train, y_val = split_train_val_data(X, y)

# Scale Inputs
X_train_scaled, X_val_scaled, input_scalers = scale_inputs(X_train, X_val)
# Scale Targets
y_train_scaled, y_val_scaled, target_scalers = scale_targets(y_train, y_val)

In [10]:
# Correctness of Data Dimensions before Training
print(X.shape)  # X shape: (samples, 120, 3)
print(y.shape)  # y shape: (samples, 9)

(45428, 120, 6)
(45428, 9)


In [11]:
model = Sequential([
    Input(shape=(120, len(INPUT_FEATURES))),

    LSTM(64, return_sequences=True, recurrent_dropout=0.1),
    Dropout(0.2),

    LSTM(32, return_sequences=False, recurrent_dropout=0.1),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(9)
])


In [12]:
model.compile(
    optimizer=Adam(learning_rate=OFFLINE_LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=13,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,  # LR is reduced by this factor
    patience=5,  # wait 5 epochs without improvement
    min_lr=1e-6,  # don't go below this LR
    verbose=1
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=1,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop, reduce_lr]
)

1136/1136 ━━━━━━━━━━━━━━━━━━━━ 96s 81ms/step - loss: 0.2020 - mae: 0.2421 - val_loss: 0.5234 - val_mae: 0.5292 - learning_rate: 0.0010


In [15]:
# Save the trained model
model.save("../Models&Scalers/LSTM/model.keras")

# ---- Target scaler metadata ----
target_meta = {
    "target_features": TARGET_FEATURES,
    "horizons": list(HORIZONS.keys()),
    "order": "feature_major"
}

# ---- Input scaler metadata ----
input_meta = {
    "input_features": INPUT_FEATURES,
    "scaled_features": SCALED_INPUT_FEATURES,
    "unscaled_features": UNSCALED_INPUT_FEATURES
}

# ---- Outlier handling metadata ----
outlier_meta = {
    "method": "iqr_capping",
    "columns": OUTLIER_COLUMNS,
    "bounds": cap_iqr_bounds
}

# ---- Save everything together ----
joblib.dump(
    {
        "target_scalers": target_scalers,
        "target_meta": target_meta,
        "input_scalers": input_scalers,
        "input_meta": input_meta,
        "outlier_meta": outlier_meta
    },
    "../Models&Scalers/LSTM/scaler_bundle.pkl"
)


['../SavedModels/LSTM_Offline/scaler_bundle.pkl']